## Load new data and assess gene overlap with DepMap

New gene dependency data isn't guaranteed to cover the same genes, use the
same gene identifiers, or list them in the same order as the DepMap data
BioBombe was trained on.
This notebook loads the new dataset and reports exactly how it lines up
against DepMap and against the trained ensemble's gene set, before anything
downstream trusts a join between them.

Run this script with `8.apply-biobombe-new-data/` as the working directory.

In [1]:
import pathlib
import sys

import pandas as pd

sys.path.insert(0, "utils")
import data_prep as dp

In [2]:
data_directory = pathlib.Path("../0.data-download/data").resolve()
model_save_dir = pathlib.Path("../3.run-biobombe/saved_models").resolve()

# Point this at the new dataset once it's available; see the module README.
NF1_data_path = pathlib.Path("data/largaespada/NF1_data.parquet")

# Everything derived from the new dataset itself, not just DepMap's public
# data, stays under data/ (gitignored) rather than the committed results/.
results_dir = pathlib.Path("data/largaespada/results")
results_dir.mkdir(parents=True, exist_ok=True)

In [3]:
depmap_df, gene_dict_df = dp.load_depmap_reference(data_directory)
trained_gene_order = dp.get_trained_gene_order(model_save_dir)

print(f"DepMap reference: {depmap_df.shape[0]} models, {depmap_df.shape[1] - 1} genes")
print(f"Trained ensemble gene set: {len(trained_gene_order)} genes")

DepMap reference: 1150 models, 18443 genes
Trained ensemble gene set: 2718 genes


/home/gway/miniconda3/envs/gene_dependency_representations/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.6.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
new_df = dp.load_new_dependency_data(NF1_data_path)
print(f"New data: {new_df.shape[0]} samples, {new_df.shape[1] - 1} genes")
new_df.head()

New data: 8 samples, 19113 genes


,ModelID,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
0,S462,-0.011443,0.219752,0.337979,-0.094739,0.156391,0.698723,0.176773,0.986754,0.402507,...,-0.554281,0.068870,0.163617,-0.191664,-0.375275,-0.242805,0.207710,-0.766841,-0.468900,-0.673073
1,ST88-14,0.053424,0.378282,0.475694,0.141624,0.529612,0.294258,-0.024992,-0.177733,0.336966,...,-0.037685,-0.500500,0.772608,0.499162,0.304846,0.125425,0.019924,-0.144481,0.019904,0.092602
2,STS-26T,-0.096644,0.457998,0.469058,0.051712,0.221865,0.469499,0.238008,-0.208519,0.102496,...,-0.104365,-2.472294,0.118021,0.320630,0.097100,0.449901,0.077049,-0.131774,0.501008,0.331042
3,iHSC1L_C3,0.219738,0.342420,0.036973,0.089192,0.117734,0.365912,0.234144,0.022396,0.205032,...,-0.360858,-0.399856,0.185544,0.309514,0.124483,0.118254,0.241866,0.116150,0.202550,-0.269800
4,iHSC1L_F12,0.199206,0.701454,0.007393,0.071413,0.087726,0.315118,0.206756,-0.007167,0.198649,...,-0.398945,-0.516269,0.327684,0.267505,0.109614,0.114930,0.229296,0.068706,0.182231,-0.318523


In [5]:
overlap_report = dp.assess_gene_overlap(new_df, depmap_df, gene_dict_df, trained_gene_order)

print(f"Resolved {overlap_report['n_resolved']} / {overlap_report['n_new_genes']} new-data gene columns to an entrez id")
print(f"Shared with DepMap's full gene set: {overlap_report['n_shared_with_depmap']} / {len(overlap_report['depmap_entrez_ids'])}")
print(f"Shared with the trained ensemble's gene set: {overlap_report['n_shared_with_trained']} / {len(overlap_report['trained_entrez_ids'])}")
print(f"Trained genes with no match in the new data: {len(overlap_report['trained_genes_missing_in_new'])}")
print(f"New data already in DepMap gene order: {overlap_report['is_order_aligned']}")

if overlap_report["unresolved_columns"]:
    print(f"\n{len(overlap_report['unresolved_columns'])} new-data columns couldn't be resolved to an entrez id, "
          "first 10:")
    print(overlap_report["unresolved_columns"][:10])

if overlap_report["trained_genes_missing_in_new"]:
    coverage = 1 - len(overlap_report["trained_genes_missing_in_new"]) / len(trained_gene_order)
    print(f"\nTrained-gene coverage: {coverage:.1%}. "
          "Missing genes get imputed with the training mean in step 2, not dropped.")

Resolved 17498 / 19113 new-data gene columns to an entrez id
Shared with DepMap's full gene set: 17498 / 18443
Shared with the trained ensemble's gene set: 2706 / 2718
Trained genes with no match in the new data: 12
New data already in DepMap gene order: False

1615 new-data columns couldn't be resolved to an entrez id, first 10:
['AAED1', 'ABCF2', 'ABO', 'ACPP', 'ACPT', 'ACRC', 'ADCK3', 'ADCK4', 'ADGB', 'ADGRF2']

Trained-gene coverage: 99.6%. Missing genes get imputed with the training mean in step 2, not dropped.


In [6]:
# Save the full report for the later steps and for manual review.
# entrez_map and the two gene-name lists are the parts worth keeping on disk;
# the entrez id sets are large and easily recomputed, so they're left out.
report_to_save = pd.DataFrame({
    "NF1_data_column": list(overlap_report["entrez_map"].keys()),
    "resolved_entrez_id": list(overlap_report["entrez_map"].values()),
})
report_to_save.to_parquet(results_dir / "gene_overlap_report.parquet", index=False)

missing_genes_df = pd.DataFrame({"trained_gene_missing_in_new": overlap_report["trained_genes_missing_in_new"]})
missing_genes_df.to_parquet(results_dir / "trained_genes_missing_in_new.parquet", index=False)

print(f"Saved gene overlap report to {results_dir / 'gene_overlap_report.parquet'}")

Saved gene overlap report to data/largaespada/results/gene_overlap_report.parquet
